In [2]:
from utils.file_utils import get_tickets_as_df

# Data exploration

In [11]:
tickets_df = get_tickets_as_df()
tickets_df.head(100)

,ticket_number,ticket_subject,sent_by,ticket_created_on,message_created_on,message,tags
0,1,NaN,Customer,2022-01-18 01:29:01.238861,2022-01-18 01:29:01.238861,"Hello, world!",Testing
1,1,NaN,Customer,2022-01-18 01:29:01.238861,2022-01-18 01:31:08.914884,Is anybody out there?,Testing
2,1,NaN,Agent,2022-01-18 01:29:01.238861,2022-01-18 01:37:46.365888,Ground control to Major Tom,Testing
3,2,NaN,Customer,2022-01-18 04:55:15.792937,2022-01-18 04:55:15.792937,whoa... chat inception,Testing
4,2,NaN,Agent,2022-01-18 04:55:15.792937,2022-01-18 04:55:35.456387,Talking to myself,Testing
...,...,...,...,...,...,...,...
95,31,NaN,Agent,2022-01-31 15:52:37.698712,2022-01-31 15:52:51.080545,df,Testing
96,31,NaN,Customer,2022-01-31 15:52:37.698712,2022-01-31 15:52:53.08065,asd,Testing
97,31,NaN,Customer,2022-01-31 15:52:37.698712,2022-01-31 15:52:53.236093,f,Testing
98,31,NaN,Customer,2022-01-31 15:52:37.698712,2022-01-31 15:52:53.376098,sdf,Testing


In [10]:
# Just investigate the tags, need to check if one ticket has multiple tags
tickets_df.groupby("ticket_number")["tags"].unique()
# conclusion:
# one ticket has multiple tags

ticket_number
1                                            [Testing]
2                                            [Testing]
3                                            [Testing]
4                                            [Testing]
5                                            [Testing]
                             ...                      
1525    [Bug,Ticketing - Email, Ticketing - Email,Bug]
1526                                         [Testing]
1527      [Product Question,Ticketing - Slack connect]
1528      [Bug,Customer Profile, Customer Profile,Bug]
1529                                         [Testing]
Name: tags, Length: 1339, dtype: object

In [13]:
# just investigate the tags and the counts
tickets_df["tags"].value_counts()
# conclusion:
# there's a lot of tags with low frequency
# so they are not very useful
# they can be dropped

tags
Testing                                                           1342
Bug                                                                887
Product Question                                                   847
Feature Request                                                    707
Sales                                                              467
                                                                  ... 
Data - Developer API,Integrations - Slack,Bug,Product Question       1
Bug,Product Question,Data - Developer API,Integrations - Slack       1
Product Question,Data - Developer API,Integrations - Slack,Bug       1
Session Recording - Player,Product Question                          1
Product Question,Ticketing - Slack connect                           1
Name: count, Length: 255, dtype: int64

# Feature engineering

In [27]:
cleaned_tickets_df = tickets_df[
    ~tickets_df["tags"].isnull()
]  # remove tags rows with null

cleaned_tickets_df = cleaned_tickets_df[
    ~cleaned_tickets_df["message"].isnull()
]  # remove message rows with null

cleaned_tickets_df = cleaned_tickets_df[
    cleaned_tickets_df["tags"].str.contains("Spam")
    | cleaned_tickets_df["tags"].str.contains("Bug")
    | cleaned_tickets_df["tags"].str.contains("Product Question")
    | cleaned_tickets_df["tags"].str.contains("Feature Request")
    | cleaned_tickets_df["tags"].str.contains("Sales")
]  # filter out spam, bug, product question, feature request, sales


cleaned_tickets_df["tags"].value_counts()
# we still can see a lot of tags with low frequency

tags
Bug                                                               870
Product Question                                                  840
Feature Request                                                   697
Sales                                                             467
Ticketing - Chat,Feature Request                                   75
                                                                 ... 
Spam,Testing                                                        1
Bug,Session Recording - Player                                      1
Bug,Integrations - Slack,Data - Developer API,Product Question      1
Product Question,Zeus                                               1
Notifications,Feature Request                                       1
Name: count, Length: 222, dtype: int64

In [30]:
def map_categories(cat: str) -> str:
    if "Bug" in cat:
        return "Bug"
    elif "Product Question" in cat:
        return "Product Question"
    elif "Feature Request" in cat:
        return "Feature Request"
    elif "Sales" in cat:
        return "Sales"
    elif "Spam" in cat:
        return "Spam"
    else:
        return "Other"


cleaned_tickets_df["tags"] = cleaned_tickets_df["tags"].apply(map_categories)
cleaned_tickets_df["tags"].value_counts()
# we can see that we only have 5 categories
# and they are: Bug, Product Question, Feature Request, Sales, Spam

tags
Bug                 1526
Feature Request     1495
Product Question    1488
Sales                477
Spam                  53
Name: count, dtype: int64

In [31]:
cleaned_tickets_df.head()

,ticket_number,ticket_subject,sent_by,ticket_created_on,message_created_on,message,tags
53,21,NaN,Customer,2022-01-24 14:19:01.51779,2022-01-24 14:19:01.51779,Hi! What is your pricing? Can I talk with some...,Sales
54,21,NaN,Agent,2022-01-24 14:19:01.51779,2022-01-24 14:19:27.936193,"Hi! Yes, we'd love to hop on a phone call! Whe...",Sales
55,22,NaN,Customer,2022-01-24 17:52:10.798997,2022-01-24 17:52:10.798997,"Hi, how much is Atlas?",Sales
56,22,NaN,Agent,2022-01-24 17:52:10.798997,2022-01-24 17:53:03.397358,"Looks like it, yeah.",Sales
77,30,NaN,Customer,2022-01-28 14:53:47.113617,2022-01-28 14:53:47.113617,hey not super urgent but fyi in ramp bank you ...,Feature Request


# Save to csv

In [32]:
cleaned_tickets_df.to_csv("data/cleaned_tickets_v1.csv", index=False)